# Python chart tour

4 November 2026 · Lecture 1

Run Python cells from the repository root. Prepare local data with `uv run python scripts/fetch_data.py`. Paths are relative to that root.

## About the data

`us_births.csv` records daily US births from 2000 to 2014. Each row is one date, with its weekday and birth count. FiveThirtyEight published these Social Security Administration counts. They let us investigate weekday patterns and changes over time. These are historical administrative counts, not a random sample of individual mothers or an explanation of why a birth occurred on a particular day.

Sources, licences, selection rules and limitations are in the [data documentation](../../data/README.md). Files are prepared by `scripts/fetch_data.py`; this notebook reads local data only.

## A question before a chart

A public-health researcher asks whether daily births are evenly spread across the week.
Start with the longer time trend, then compare weekdays. What might a single average hide?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# parse_dates asks pandas to read "date" as dates instead of text.
births = pd.read_csv("data/raw/us_births.csv", parse_dates=["date"])
print(births.head())
# isna() marks missing values; sum() counts them in each column.
missing_values = births.isna()
missing_counts = missing_values.sum()
print(missing_counts)
print("Dates:", births["date"].min(), births["date"].max())

## From a date column to a DatetimeIndex

An index supplies the row labels on the left of a pandas table. Our table currently
uses numbers. `parse_dates` converted the **values in the date column**, not the index.

In [ ]:
print(births.index[:3])
print(births["date"].dtype)

### 1. Use dates as row labels

`set_index` returns a table with `date` as its index instead of an ordinary column.
Because these values are datetimes, pandas creates a **DatetimeIndex**.
No rows have been combined: each row is still one day. The original `births` stays unchanged.

In [ ]:
births_by_date = births.set_index("date")
print(births_by_date.head())
print(births_by_date.index[:3])

### 2. Put the dates in order and select a month

`sort_index` orders rows by their date labels. With a DatetimeIndex, `.loc["2014-01"]`
selects all dates in January 2014, not just 1 January. Selection is not aggregation.

In [ ]:
births_by_date = births_by_date.sort_index()
january_2014 = births_by_date.loc["2014-01"]
print(january_2014.head())
print("Daily rows in January 2014:", len(january_2014))

### 3. Keep the counts and their date labels

Selecting the `births` column gives a Series: one count per day, with the same date index.
We use the full period below, not just January 2014.

In [ ]:
daily_births = births_by_date["births"]
print(daily_births.head())

[pandas reference: time series and date indexes](https://pandas.pydata.org/docs/user_guide/timeseries.html).

## Line plots: change over time

Calculate the mean number of births per day in each month. Monthly totals also depend
on the number of days in a month; this mean answers a different, explicit question.
First define the monthly groups, then choose how to summarise them.

In [ ]:
# MS means month-start: each calendar-month group is labelled by its first day.
monthly_groups = daily_births.resample("MS")
# size() shows how many daily rows belong to each group, not how many births.
days_per_month = monthly_groups.size()
print(days_per_month.head())

Now calculate the mean **within each group**. A label such as `2000-01-01` identifies
the whole month's average; it is not the birth count on 1 January.

In [ ]:
monthly = monthly_groups.mean()
print(monthly.head())

Plot the monthly result. Its index supplies the dates; its values supply the averages.

In [ ]:
# fig is the whole figure; ax is its plotting area. figsize is in inches.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly.index, monthly.values)
ax.set(title="US births per day, averaged within each month", xlabel="Month", ylabel="Births per day")
# Rotate and align date labels so that they are easier to read.
fig.autofmt_xdate()
# Adjust spacing to make room for titles and axis labels.
fig.tight_layout()
plt.show()

## Bar plots: compare weekdays

Each bar represents a mean across dates, not a total across all Mondays or Tuesdays.
Keep the familiar weekday order and start bar lengths at zero. Before running: do you expect weekends to differ?

In [ ]:
# Define the groups, select the quantity, then calculate its mean.
weekday_groups = births.groupby("weekday")
weekday_births = weekday_groups["births"]
weekday_means = weekday_births.mean()
print(weekday_means)

In [ ]:
# reindex changes the display order, not the averages.
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = weekday_means.reindex(day_order)
print(by_day)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_day.index, by_day.values)
# ylim fixes the vertical range; a zero baseline makes bar lengths comparable.
ax.set(title="Average daily births by weekday, 2000–2014", ylabel="Births per day", ylim=(0, 14000))
# Rotate the x-axis labels by 25 degrees to avoid crowding.
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
plt.show()
print(by_day.round(1))

The weekend difference is visible in these records. Scheduling of deliveries is one
possible explanation, but these counts alone cannot distinguish delivery methods or establish a cause.

## Scatter, histogram, box, violin and strip plots

Use the same 365 dates from 2014 throughout this tour. We are comparing daily counts,
not showing individual births. Which views reveal the weekly pattern most clearly?

In [ ]:
# Use the same 2014 sample in every plot; copy() keeps it separate from births.
in_2014 = births["year"] == 2014
tour = births.loc[in_2014].copy()
print("Dates shown:", len(tour))
# count() counts non-missing birth values within each weekday group.
tour_groups = tour.groupby("weekday")
tour_births = tour_groups["births"]
tour_counts = tour_births.count()
tour_counts = tour_counts.reindex(day_order)
print(tour_counts)

### Scatterplot

Each point is one date. Unlike the monthly line, the daily points reveal short-run variation.

In [ ]:
# .dt accesses date features; dayofyear gives 1 for 1 January, etc.
day_number = tour["date"].dt.dayofyear
fig, ax = plt.subplots(figsize=(9, 4))
# alpha controls opacity (0 transparent, 1 opaque); s sets marker area.
ax.scatter(day_number, tour["births"], alpha=0.6, s=15)
ax.set(title="Daily US births in 2014", xlabel="Day of year", ylabel="Births")
fig.tight_layout()
plt.show()

### Histogram

A histogram counts dates in numerical intervals. The two concentrations suggest that
pooling all days may hide a meaningful grouping. We can investigate it rather than merely choose a colour.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
# bins=20 divides the observed range into 20 equal-width intervals.
# Each bar counts dates in an interval; white edges separate the bars.
ax.hist(tour["births"], bins=20, edgecolor="white")
ax.set(title="Distribution of daily US births in 2014", xlabel="Births per day", ylabel="Number of dates")
fig.tight_layout()
plt.show()

### Box plot

The box covers the middle half of daily counts, with a line at the median. Whiskers reach the furthest observations within 1.5 box-lengths. Points beyond them are not automatically errors.

In [ ]:
# Fix the random seed so the strip plot's jitter repeats on each run.
np.random.seed(42)
fig, ax = plt.subplots(figsize=(9, 4.5))
# Seaborn uses column names for x/y; order preserves calendar order.
# Each box shows the middle half of one weekday's birth counts.
# The line inside each box marks the median.
sns.boxplot(data=tour, x="weekday", y="births", order=day_order, color="#9DBFCA", ax=ax)
# Use the same vertical limits in all three views for fair comparison.
ax.set(title="Daily US births by weekday: box plot", xlabel="", ylabel="Births per day", ylim=(5000, 16000))
ax.tick_params(axis="x", rotation=25)
# Figure text coordinates run from 0 to 1; (0.01, 0.01) is near bottom-left.
fig.text(0.01, 0.01, "Source: SSA via FiveThirtyEight. All 365 dates in 2014; same sample in each view.", fontsize=9)
# rect=(left, bottom, right, top) reserves the bottom 8% for the note.
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

### Violin plot

Width shows a smoothed concentration of daily counts. `cut=0` stops at the observed range. `density_norm="width"` gives equal maximum widths, so width does not compare sample sizes. A violin is not a confidence interval.

In [ ]:
# Fix the random seed so the strip plot's jitter repeats on each run.
np.random.seed(42)
fig, ax = plt.subplots(figsize=(9, 4.5))
# Seaborn uses column names for x/y; order preserves calendar order.
# inner="quart" adds lines at the quartiles, including the median.
# cut=0 keeps the violin within the observed range of each group.
# density_norm="width" gives each violin the same maximum width.
sns.violinplot(data=tour, x="weekday", y="births", order=day_order, inner="quart", cut=0, density_norm="width", color="#9DBFCA", ax=ax)
# Use the same vertical limits in all three views for fair comparison.
ax.set(title="Daily US births by weekday: violin plot", xlabel="", ylabel="Births per day", ylim=(5000, 16000))
ax.tick_params(axis="x", rotation=25)
# Figure text coordinates run from 0 to 1; (0.01, 0.01) is near bottom-left.
fig.text(0.01, 0.01, "Source: SSA via FiveThirtyEight. All 365 dates in 2014; same sample in each view.", fontsize=9)
# rect=(left, bottom, right, top) reserves the bottom 8% for the note.
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

### Strip plot

Each dot is a date. Sideways jitter reduces overlap and has no substantive meaning. Compare the actual points with the smoothed violin.

In [ ]:
# Fix the random seed so the strip plot's jitter repeats on each run.
np.random.seed(42)
fig, ax = plt.subplots(figsize=(9, 4.5))
# Seaborn uses column names for x/y; order preserves calendar order.
# jitter adds random sideways movement to reduce overlapping dots.
# alpha=0.5 makes dots partly transparent; size=3 sets their size.
sns.stripplot(data=tour, x="weekday", y="births", order=day_order, jitter=0.2, alpha=0.5, size=3, color="#176B73", ax=ax)
# Use the same vertical limits in all three views for fair comparison.
ax.set(title="Daily US births by weekday: strip plot", xlabel="", ylabel="Births per day", ylim=(5000, 16000))
ax.tick_params(axis="x", rotation=25)
# Figure text coordinates run from 0 to 1; (0.01, 0.01) is near bottom-left.
fig.text(0.01, 0.01, "Source: SSA via FiveThirtyEight. All 365 dates in 2014; same sample in each view.", fontsize=9)
# rect=(left, bottom, right, top) reserves the bottom 8% for the note.
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

What does each view reveal or conceal? Check whether the fixed limits contain all observations.
How would you explain a violin to a reader who has never seen one?
Session 2 develops distribution comparisons and distinguishes them from uncertainty about estimates.

## Labels, notes and export

The title states the quantity. Labels give units. The note identifies the source,
period and aggregation. Keep the note attached when exporting the Figure.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly.index, monthly.values)
ax.set(title="US births per day, averaged within each month", xlabel="Month", ylabel="Births per day")
fig.autofmt_xdate()
# Add a note to the figure itself, so it stays with the exported image.
fig.text(0.01, 0.01, "SSA via FiveThirtyEight, 2000–2014. Mean daily counts within calendar months; no dates excluded.", fontsize=9)
# Leave space below the axes for the figure note.
fig.tight_layout(rect=(0, 0.09, 1, 1))
# Save a PNG: dpi sets resolution; bbox_inches="tight" trims excess margins.
fig.savefig("figures/01_monthly_births.png", dpi=150, bbox_inches="tight")
plt.show()

## Your redesign

Keep the final 20 minutes for [the exercise](exercises.ipynb). It supplies a pie chart
of the 12 monthly birth totals in 2014. The data preparation is provided.

Course AI use and disclosure requirements are in [the policy](../../docs/ai_policy.md). If you use assistance, verify the code, numbers and interpretation and disclose it as required.